# Week 3 — MLflow Forecasting Pipeline — Ona

### Goal
Build a reproducible machine-learning forecasting pipeline for `unit_sales`.

This notebook continues the Week 2 statistical modelling work and prepares the project for the Week 3 submission:

| Step | What happens |
|---|---|
| 1 | Install and import libraries |
| 2 | Load the feature-engineered dataset |
| 3 | Quick EDA check |
| 4 | Feature engineering for machine learning |
| 5 | Chronological train/test split |
| 6 | Train and evaluate several models |
| 7 | Tune XGBoost with time-series cross-validation |
| 8 | Select and save the best model |
| 9 | Log experiments with MLflow |

The output of this notebook is used by the Streamlit notebook/app.


---
## Step 1 — Install Libraries

This cell installs the required packages quietly. If your environment already has them, it will finish quickly.

For GitHub, the same packages should also be listed in `requirements.txt`.


In [ ]:
import sys

# Quiet installation keeps the notebook output clean.
!{sys.executable} -m pip install -q pandas numpy matplotlib seaborn scikit-learn statsmodels prophet xgboost mlflow joblib plotly ipywidgets


---
## Step 2 — Import Libraries


In [ ]:
from pathlib import Path
import json
import time
import warnings

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit

from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.holtwinters import ExponentialSmoothing

from prophet import Prophet

try:
    import xgboost as xgb
    XGBOOST_AVAILABLE = True
except Exception as e:
    XGBOOST_AVAILABLE = False
    print("XGBoost could not be imported:", e)

import mlflow

warnings.filterwarnings("ignore")
plt.rcParams["figure.figsize"] = (14, 4)

print("All libraries imported successfully!")


---
## Step 3 — Set Project Paths

Recommended repository structure:

```text
time_series_analysis_ona/
├── notebooks/
│   └── W3-mlflow_ona.ipynb
├── outputs/
│   └── timeseries_with_features.csv
├── models/
├── mlruns/
├── README.md
└── requirements.txt
```

This notebook searches common folders automatically, so it can work both locally and from GitHub.


In [ ]:
# Current notebook folder is usually notebooks/.
CURRENT_DIR = Path.cwd()

# If we are inside notebooks/ or preparation/, the project root is one level up.
if CURRENT_DIR.name.lower() in {"notebooks", "preparation"}:
    PROJECT_DIR = CURRENT_DIR.parent
else:
    PROJECT_DIR = CURRENT_DIR

DATA_FILE_NAME = "timeseries_with_features.csv"

# Search in the most common project folders.
candidate_paths = [
    PROJECT_DIR / "outputs" / DATA_FILE_NAME,
    PROJECT_DIR / "data" / DATA_FILE_NAME,
    CURRENT_DIR / "outputs" / DATA_FILE_NAME,
    CURRENT_DIR / "data" / DATA_FILE_NAME,
    CURRENT_DIR / DATA_FILE_NAME,
]

DATA_PATH = next((path for path in candidate_paths if path.exists()), None)

MODELS_DIR = PROJECT_DIR / "models"
MLRUNS_DIR = PROJECT_DIR / "mlruns"
REPORTS_DIR = PROJECT_DIR / "reports"

MODELS_DIR.mkdir(exist_ok=True)
MLRUNS_DIR.mkdir(exist_ok=True)
REPORTS_DIR.mkdir(exist_ok=True)

print("PROJECT_DIR:", PROJECT_DIR)
print("MODELS_DIR: ", MODELS_DIR)
print("MLRUNS_DIR: ", MLRUNS_DIR)
print("REPORTS_DIR:", REPORTS_DIR)

if DATA_PATH is None:
    print("
Could not find the dataset automatically.")
    print("Searched here:")
    for path in candidate_paths:
        print(" -", path)
    raise FileNotFoundError("Place timeseries_with_features.csv in outputs/ or data/ and run again.")
else:
    print("
Loading file:", DATA_PATH)


---
## Step 4 — Load the Dataset

The dataset should be the feature-engineered CSV created in the previous notebook.


In [ ]:
df = pd.read_csv(DATA_PATH)

# Standardize date column.
if "date" not in df.columns:
    raise ValueError("The dataset must contain a 'date' column.")

df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").reset_index(drop=True)

print("Dataset loaded successfully!")
print("Shape:", df.shape)
print("Date range:", df["date"].min().date(), "to", df["date"].max().date())

display(df.head())


---
## Step 5 — Quick Data Quality Check

We check for missing dates and missing values before modelling.


In [ ]:
full_range = pd.date_range(df["date"].min(), df["date"].max(), freq="D")
missing_dates = full_range.difference(df["date"])

print("Expected daily dates:", len(full_range))
print("Rows in dataset:      ", len(df))
print("Missing dates:        ", len(missing_dates))

missing_values = df.isna().sum()
missing_values = missing_values[missing_values > 0]

print("
Missing values:")
if missing_values.empty:
    print("No missing values found.")
else:
    display(missing_values)


If dates are missing, we reindex to a complete daily calendar. This keeps the time series regular, which is important for forecasting models.


In [ ]:
df = df.set_index("date").reindex(full_range)
df.index.name = "date"

# Fill target carefully. Interpolation is used only for missing dates, not to change existing values.
df["unit_sales"] = df["unit_sales"].interpolate(method="linear").ffill().bfill()

# Fill other feature columns if any missing values remain.
df = df.ffill().bfill()

# Keep date also as a column for Prophet and plotting.
df = df.reset_index()

print("After cleaning:")
print("Rows:", len(df))
print("Remaining missing values:", int(df.isna().sum().sum()))


---
## Step 6 — Exploratory Data Analysis

This is a short EDA section because the detailed EDA was already done in previous notebooks. Here we only verify the target behavior before machine-learning modelling.


In [ ]:
plt.figure(figsize=(14, 4))
plt.plot(df["date"], df["unit_sales"], linewidth=1.2)
plt.title("Daily Unit Sales Over Time")
plt.xlabel("Date")
plt.ylabel("Unit Sales")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Unit sales summary:")
display(df["unit_sales"].describe().round(2))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(df["unit_sales"], bins=30, edgecolor="white")
axes[0].set_title("Distribution of Unit Sales")
axes[0].set_xlabel("Unit Sales")
axes[0].set_ylabel("Count")
axes[0].grid(True, alpha=0.3)

axes[1].boxplot(df["unit_sales"], vert=True, patch_artist=True)
axes[1].set_title("Boxplot of Unit Sales")
axes[1].set_ylabel("Unit Sales")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


### EDA interpretation

- The time series contains regular fluctuations and occasional sales spikes.
- The distribution is right-skewed, meaning a few days have much higher demand than usual.
- These patterns suggest that models using lag features, rolling statistics, and calendar variables may perform better than models using the target alone.


---
## Step 7 — Feature Engineering for Machine Learning

Tree-based models such as Random Forest and XGBoost do not understand temporal order automatically. We give them time-series memory through lag, rolling, and calendar features.


In [ ]:
def create_ml_features(data: pd.DataFrame) -> pd.DataFrame:
    """Create or standardize features used by machine-learning models."""
    data = data.copy()
    data["date"] = pd.to_datetime(data["date"])
    data = data.sort_values("date").reset_index(drop=True)

    # Calendar features
    data["year"] = data["date"].dt.year
    data["month"] = data["date"].dt.month
    data["day"] = data["date"].dt.day
    data["dayofweek"] = data["date"].dt.dayofweek
    data["quarter"] = data["date"].dt.quarter
    data["week_of_year"] = data["date"].dt.isocalendar().week.astype(int)
    data["is_weekend"] = (data["dayofweek"] >= 5).astype(int)
    data["is_month_start"] = data["date"].dt.is_month_start.astype(int)
    data["is_month_end"] = data["date"].dt.is_month_end.astype(int)

    # Time-series memory features. Use shift(1) for rolling features to avoid leakage.
    for lag in [1, 7, 14, 30]:
        col = f"lag_{lag}"
        if col not in data.columns:
            data[col] = data["unit_sales"].shift(lag)

    if "rolling_7d_mean" not in data.columns:
        data["rolling_7d_mean"] = data["unit_sales"].shift(1).rolling(7).mean()
    if "rolling_14d_mean" not in data.columns:
        data["rolling_14d_mean"] = data["unit_sales"].shift(1).rolling(14).mean()
    if "rolling_30d_mean" not in data.columns:
        data["rolling_30d_mean"] = data["unit_sales"].shift(1).rolling(30).mean()
    if "rolling_7d_std" not in data.columns:
        data["rolling_7d_std"] = data["unit_sales"].shift(1).rolling(7).std()

    # Convert booleans to integers for modelling.
    for col in data.select_dtypes(include=["bool"]).columns:
        data[col] = data[col].astype(int)

    return data

feature_df = create_ml_features(df)

# Drop rows created by lag/rolling features.
feature_df = feature_df.dropna().reset_index(drop=True)

print("Feature dataset shape:", feature_df.shape)
print("Date range after feature engineering:", feature_df["date"].min().date(), "to", feature_df["date"].max().date())
display(feature_df.head())


---
## Step 8 — Chronological Train/Test Split

We do **not** use a random split. In time series, the model must train on the past and test on the future.

The test set is the last 89 days, matching the Week 2 notebook.


In [ ]:
TEST_SIZE = 89

train_df = feature_df.iloc[:-TEST_SIZE].copy()
test_df = feature_df.iloc[-TEST_SIZE:].copy()

print("Training rows:", len(train_df), "|", train_df["date"].min().date(), "to", train_df["date"].max().date())
print("Test rows:    ", len(test_df),  "|", test_df["date"].min().date(),  "to", test_df["date"].max().date())


In [ ]:
plt.figure(figsize=(14, 4))
plt.plot(train_df["date"], train_df["unit_sales"], label="Train")
plt.plot(test_df["date"], test_df["unit_sales"], label="Test")
plt.axvline(test_df["date"].min(), linestyle="--", alpha=0.7, label="Split")
plt.title("Chronological Train/Test Split")
plt.xlabel("Date")
plt.ylabel("Unit Sales")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


---
## Step 9 — Prepare Feature Matrix

We keep only numeric columns and remove columns that should not be used as features.


In [ ]:
TARGET = "unit_sales"
EXCLUDE_COLUMNS = {
    "date", TARGET,
    "day_of_week",       # string version of weekday, if present
    "is_outlier",        # target-derived flag; can leak target behavior
    "zscore",            # target-derived diagnostic; not used for modelling
}

numeric_columns = feature_df.select_dtypes(include=[np.number]).columns.tolist()
FEATURES = [col for col in numeric_columns if col not in EXCLUDE_COLUMNS]

X_train = train_df[FEATURES]
y_train = train_df[TARGET]
X_test = test_df[FEATURES]
y_test = test_df[TARGET]

print("Number of features:", len(FEATURES))
print("Features used:")
print(FEATURES)

# Final safety check
print("
NaNs in X_train:", int(X_train.isna().sum().sum()))
print("NaNs in X_test: ", int(X_test.isna().sum().sum()))


---
## Step 10 — Evaluation Function

We use the same metrics for all models so the comparison is fair.


In [ ]:
def evaluate_model(name, actual, predicted, training_time=None):
    """Return common forecasting metrics as a dictionary."""
    actual = np.asarray(actual, dtype=float).ravel()
    predicted = np.asarray(predicted, dtype=float).ravel()

    mae = mean_absolute_error(actual, predicted)
    rmse = np.sqrt(mean_squared_error(actual, predicted))
    bias = np.mean(predicted - actual)
    r2 = r2_score(actual, predicted)

    mask = actual != 0
    mape = np.mean(np.abs((actual[mask] - predicted[mask]) / actual[mask])) * 100 if mask.any() else np.nan

    result = {
        "model": name,
        "MAE": round(mae, 2),
        "RMSE": round(rmse, 2),
        "MAPE": round(mape, 2),
        "Bias": round(bias, 2),
        "R2": round(r2, 4),
        "Training_time_s": round(training_time, 2) if training_time is not None else None,
    }

    print(f"{name}")
    print(f"  MAE:  {result['MAE']}")
    print(f"  RMSE: {result['RMSE']}")
    print(f"  MAPE: {result['MAPE']}%")
    print(f"  Bias: {result['Bias']}")
    print(f"  R²:   {result['R2']}")
    if training_time is not None:
        print(f"  Training time: {result['Training_time_s']} seconds")
    print()

    return result

results = []
predictions = {}
model_objects = {}

print("Evaluation function ready.")


---
## Step 11 — Baseline Model

A simple seasonal naive baseline predicts each day using the value from the same day of the previous week. More advanced models should beat this baseline.


In [ ]:
# Seasonal naive forecast: use lag_7 already aligned in the test set.
naive_preds = test_df["lag_7"].values

results.append(evaluate_model("Seasonal Naive", y_test.values, naive_preds))
predictions["Seasonal Naive"] = naive_preds


---
## Step 12 — Train Statistical Models

These models are included for comparison with Week 2. The main Week 3 focus is the machine-learning model and MLflow tracking.


In [ ]:
# SARIMAX with weekly seasonality
start = time.time()
sarimax_model = SARIMAX(
    train_df.set_index("date")["unit_sales"],
    order=(1, 1, 1),
    seasonal_order=(1, 1, 1, 7),
    enforce_stationarity=False,
    enforce_invertibility=False,
)
sarimax_fit = sarimax_model.fit(disp=False)
sarimax_preds = sarimax_fit.forecast(steps=len(test_df)).values
sarimax_time = time.time() - start

results.append(evaluate_model("SARIMAX", y_test.values, sarimax_preds, sarimax_time))
predictions["SARIMAX"] = sarimax_preds
model_objects["SARIMAX"] = sarimax_fit


In [ ]:
# Holt-Winters with additive weekly seasonality.
# Additive is safer than multiplicative because the series contains zero / very low sales values.
start = time.time()
hw_model = ExponentialSmoothing(
    train_df.set_index("date")["unit_sales"],
    trend="add",
    seasonal="add",
    seasonal_periods=7,
)
hw_fit = hw_model.fit(optimized=True)
hw_preds = hw_fit.forecast(len(test_df)).values
hw_time = time.time() - start

results.append(evaluate_model("Holt-Winters", y_test.values, hw_preds, hw_time))
predictions["Holt-Winters"] = hw_preds
model_objects["Holt-Winters"] = hw_fit


In [ ]:
# Prophet requires columns named ds (date) and y (target).
prophet_train = train_df[["date", "unit_sales"]].rename(columns={"date": "ds", "unit_sales": "y"})
prophet_test = test_df[["date", "unit_sales"]].rename(columns={"date": "ds", "unit_sales": "y"})

start = time.time()
prophet_model = Prophet(weekly_seasonality=True, yearly_seasonality=True, daily_seasonality=False)
prophet_model.fit(prophet_train)
future = prophet_model.make_future_dataframe(periods=len(prophet_test), freq="D")
prophet_forecast = prophet_model.predict(future)
prophet_preds = prophet_forecast.set_index("ds").loc[prophet_test["ds"], "yhat"].values
prophet_time = time.time() - start

results.append(evaluate_model("Prophet", y_test.values, prophet_preds, prophet_time))
predictions["Prophet"] = prophet_preds
model_objects["Prophet"] = prophet_model


---
## Step 13 — Train Machine-Learning Models

Random Forest and XGBoost use the engineered features created above.


In [ ]:
start = time.time()
rf_model = RandomForestRegressor(
    n_estimators=300,
    max_depth=None,
    random_state=42,
    n_jobs=-1,
)
rf_model.fit(X_train, y_train)
rf_preds = rf_model.predict(X_test)
rf_time = time.time() - start

results.append(evaluate_model("Random Forest", y_test.values, rf_preds, rf_time))
predictions["Random Forest"] = rf_preds
model_objects["Random Forest"] = rf_model


In [ ]:
if XGBOOST_AVAILABLE:
    start = time.time()
    xgb_model = xgb.XGBRegressor(
        objective="reg:squarederror",
        n_estimators=300,
        learning_rate=0.05,
        max_depth=4,
        subsample=0.9,
        colsample_bytree=0.9,
        random_state=42,
        verbosity=0,
    )
    xgb_model.fit(X_train, y_train)
    xgb_preds = xgb_model.predict(X_test)
    xgb_time = time.time() - start

    results.append(evaluate_model("XGBoost Baseline", y_test.values, xgb_preds, xgb_time))
    predictions["XGBoost Baseline"] = xgb_preds
    model_objects["XGBoost Baseline"] = xgb_model
else:
    print("Skipping XGBoost because it is not available in this environment.")


---
## Step 14 — Hyperparameter Tuning with TimeSeriesSplit

We tune XGBoost using `TimeSeriesSplit`, so validation always respects chronological order.


In [ ]:
if XGBOOST_AVAILABLE:
    param_grid = {
        "n_estimators": [100, 300, 500],
        "max_depth": [2, 3, 4, 5],
        "learning_rate": [0.01, 0.03, 0.05, 0.1],
        "subsample": [0.7, 0.8, 1.0],
        "colsample_bytree": [0.7, 0.8, 1.0],
    }

    tscv = TimeSeriesSplit(n_splits=3)

    search = RandomizedSearchCV(
        estimator=xgb.XGBRegressor(objective="reg:squarederror", random_state=42, verbosity=0),
        param_distributions=param_grid,
        n_iter=20,
        scoring="neg_mean_absolute_error",
        cv=tscv,
        random_state=42,
        n_jobs=-1,
    )

    print("Starting XGBoost tuning...")
    start = time.time()
    search.fit(X_train, y_train)
    tuning_time = time.time() - start

    best_xgb = search.best_estimator_
    tuned_preds = best_xgb.predict(X_test)

    print("Best parameters:")
    print(search.best_params_)

    results.append(evaluate_model("XGBoost Tuned", y_test.values, tuned_preds, tuning_time))
    predictions["XGBoost Tuned"] = tuned_preds
    model_objects["XGBoost Tuned"] = best_xgb
else:
    search = None
    print("Skipping tuning because XGBoost is not available.")


---
## Step 15 — Compare Models and Select Best Model

The best model is selected by the lowest MAE because MAE is easy to interpret in sales units.


In [ ]:
results_df = pd.DataFrame(results).drop_duplicates(subset="model", keep="last").set_index("model")
results_df = results_df.sort_values("MAE")

display(results_df)

best_model_name = results_df.index[0]
best_mae = results_df.iloc[0]["MAE"]

print(f"Best model by MAE: {best_model_name}")
print(f"Best MAE: {best_mae:.2f} sales units")


In [ ]:
plt.figure(figsize=(12, 4))
results_df["MAE"].sort_values().plot(kind="bar")
plt.title("Model Comparison by MAE")
plt.ylabel("MAE")
plt.xlabel("Model")
plt.xticks(rotation=45, ha="right")
plt.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(14, 5))
plt.plot(test_df["date"], y_test.values, label="Actual", linewidth=2)

# Plot the top 3 model predictions by MAE.
for model_name in results_df.head(3).index:
    if model_name in predictions:
        plt.plot(test_df["date"], predictions[model_name], label=model_name, alpha=0.8)

plt.title("Actual vs Top Forecasts")
plt.xlabel("Date")
plt.ylabel("Unit Sales")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


---
## Step 16 — Feature Importance

Feature importance helps explain which variables the best tree-based model used most.


In [ ]:
# Use tuned XGBoost if available; otherwise use Random Forest.
importance_model_name = "XGBoost Tuned" if "XGBoost Tuned" in model_objects else "Random Forest"
importance_model = model_objects[importance_model_name]

if hasattr(importance_model, "feature_importances_"):
    importance_df = pd.DataFrame({
        "feature": FEATURES,
        "importance": importance_model.feature_importances_,
    }).sort_values("importance", ascending=False).head(15)

    display(importance_df)

    plt.figure(figsize=(10, 5))
    plt.barh(importance_df["feature"][::-1], importance_df["importance"][::-1])
    plt.title(f"Top Feature Importances — {importance_model_name}")
    plt.xlabel("Importance")
    plt.tight_layout()
    plt.show()
else:
    print(f"{importance_model_name} does not provide feature_importances_.")


---
## Step 17 — Save Best Model and Metadata

The Streamlit app will load these files from the `models/` folder.


In [ ]:
def save_model_artifacts(model_name, model_objects, models_dir):
    models_dir = Path(models_dir)
    models_dir.mkdir(exist_ok=True)

    if model_name not in model_objects:
        raise ValueError(f"Best model '{model_name}' was not saved in model_objects.")

    model_path = models_dir / "best_model.pkl"
    name_path = models_dir / "best_model_name.txt"
    features_path = models_dir / "feature_columns.json"
    metrics_path = models_dir / "model_metrics.csv"

    joblib.dump(model_objects[model_name], model_path)
    name_path.write_text(model_name)
    features_path.write_text(json.dumps(FEATURES, indent=2))
    results_df.to_csv(metrics_path)

    print("Saved:")
    print(" -", model_path)
    print(" -", name_path)
    print(" -", features_path)
    print(" -", metrics_path)

    return model_path

best_model_path = save_model_artifacts(best_model_name, model_objects, MODELS_DIR)


---
## Step 18 — Log Experiments with MLflow

MLflow stores each model run, its metrics, and the final best model information.


In [ ]:
mlflow.set_tracking_uri(MLRUNS_DIR.as_uri())
mlflow.set_experiment("ona_retail_sales_forecasting")

print("MLflow tracking URI:", MLRUNS_DIR.as_uri())
print("Experiment: ona_retail_sales_forecasting")


In [ ]:
# Log one run per model.
for model_name, row in results_df.iterrows():
    with mlflow.start_run(run_name=model_name):
        mlflow.log_param("model_name", model_name)
        mlflow.log_param("target", TARGET)
        mlflow.log_param("test_size_days", TEST_SIZE)
        mlflow.log_param("n_features", len(FEATURES))

        mlflow.log_metric("MAE", float(row["MAE"]))
        mlflow.log_metric("RMSE", float(row["RMSE"]))
        mlflow.log_metric("MAPE", float(row["MAPE"]))
        mlflow.log_metric("Bias", float(row["Bias"]))
        mlflow.log_metric("R2", float(row["R2"]))

        if model_name == "XGBoost Tuned" and search is not None:
            mlflow.log_params(search.best_params_)

print(f"Logged {len(results_df)} model runs to MLflow.")


In [ ]:
# Log the final best model run.
with mlflow.start_run(run_name=f"BEST_MODEL__{best_model_name}"):
    mlflow.log_param("best_model", best_model_name)
    mlflow.log_param("model_path", str(best_model_path))
    mlflow.log_param("feature_columns", ",".join(FEATURES))
    mlflow.log_param("test_size_days", TEST_SIZE)

    best_row = results_df.loc[best_model_name]
    mlflow.log_metric("MAE", float(best_row["MAE"]))
    mlflow.log_metric("RMSE", float(best_row["RMSE"]))
    mlflow.log_metric("MAPE", float(best_row["MAPE"]))
    mlflow.log_metric("Bias", float(best_row["Bias"]))
    mlflow.log_metric("R2", float(best_row["R2"]))

    mlflow.log_artifact(str(best_model_path))
    mlflow.log_artifact(str(MODELS_DIR / "feature_columns.json"))
    mlflow.log_artifact(str(MODELS_DIR / "model_metrics.csv"))

print("Best model logged to MLflow.")


---
## Final Summary

This notebook completed the Week 3 modelling pipeline:

- Loaded the feature-engineered time-series dataset.
- Checked data quality and temporal order.
- Added lag, rolling, and calendar features for machine learning.
- Split the data chronologically into train and test periods.
- Trained statistical and machine-learning models.
- Tuned XGBoost using time-series cross-validation.
- Selected the best model using MAE.
- Saved the best model and metadata for Streamlit.
- Logged all model runs with MLflow.

### Files created

```text
models/best_model.pkl
models/best_model_name.txt
models/feature_columns.json
models/model_metrics.csv
mlruns/
```

The next step is to use these artifacts in `streamlit.ipynb` or the Streamlit app.
